# 29th Oct

In [ ]:
# @title train_on_full_kg.py (Stage B: Model Training)
# --- 0. INSTALL DEPENDENCIES ---
!pip install -q transformers sentence-transformers==2.7.0 torch-geometric "spacy[transformers,lookups]"
!python -m spacy download en_core_web_lg

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from collections import defaultdict
import spacy
import random
import json

from sentence_transformers import SentenceTransformer
from torch_geometric.data import HeteroData
from torch_geometric.nn import RGCNConv

# --- 2. CONFIGURATION & SETUP ---
print("\n--- Configuring Paths and Environment ---")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    print("This script is designed for Google Colab.")

BASE_PATH = "/content/drive/My Drive/Secure_KGRAG_Project/"
# --- INPUTS ---
KG_INPUT_PATH = os.path.join(BASE_PATH, "full_kg_v1/")
# --- OUTPUTS ---
MODEL_OUTPUT_PATH = os.path.join(BASE_PATH, "gfm_retriever_v1/")
os.makedirs(MODEL_OUTPUT_PATH, exist_ok=True)
print(f"Trained model will be saved to: {MODEL_OUTPUT_PATH}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# --- 3. LOAD GRAPH ASSETS FROM DISK ---
print("\n--- Loading Pre-built Graph Assets ---")
try:
    # Set weights_only=False as we are loading a full Python object (HeteroData)
    graph = torch.load(os.path.join(KG_INPUT_PATH, "full_graph.pt"), weights_only=False)

    with open(os.path.join(KG_INPUT_PATH, "node_maps.json"), 'r') as f:
        node_maps = json.load(f)
    with open(os.path.join(KG_INPUT_PATH, "relations.json"), 'r') as f:
        serializable_relations = json.load(f)
        relations = {eval(k): v for k, v in serializable_relations.items()}
    with open(os.path.join(KG_INPUT_PATH, "node_counters.json"), 'r') as f:
        node_counters = json.load(f)

    print("All graph assets loaded successfully.")
    print("Graph details:"); print(graph)

except Exception as e:
    print(f"FATAL ERROR loading graph assets: {e}"); raise

# --- 4. DEFINE MODEL AND TRAINING ---
class GNNRetriever(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_relations):
        super().__init__()
        self.gnn_backbone = RGCNConv(in_channels, hidden_channels, num_relations)
        self.scoring_head = nn.Sequential(nn.Linear(hidden_channels + in_channels, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, homogeneous_graph, query_embedding, sentence_start_idx, sentence_end_idx):
        node_embeddings = F.relu(self.gnn_backbone(homogeneous_graph.x, homogeneous_graph.edge_index, homogeneous_graph.edge_type))
        sentence_embeddings = node_embeddings[sentence_start_idx:sentence_end_idx]
        query_expanded = query_embedding.unsqueeze(0).repeat(sentence_embeddings.shape[0], 1)
        fused = torch.cat([sentence_embeddings, query_expanded], dim=1)
        return self.scoring_head(fused).squeeze(-1)

print("\n--- Initiating Supervised Training with Hard Negative Mining ---")
embedding_model = SentenceTransformer('all-mpnet-base-v2')
homogeneous_graph = graph.to_homogeneous().to(DEVICE)
model = GNNRetriever(homogeneous_graph.x.shape[1], 256, len(graph.edge_types)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

training_queries = {
    "risks": ("What are the primary modern slavery risks?", "RISK"),
    "diligence": ("Show me examples of due diligence processes.", "CONTROL"),
    "governance": ("Describe the governance and oversight structures.", "GOVERNANCE")
}
query_embeddings = {q_type: embedding_model.encode(q_text, convert_to_tensor=True).to(DEVICE) for q_type, (q_text, _) in training_queries.items()}

print("\n--- Generating Triplets with Hard Negative Mining ---")
sentence_entity_map = defaultdict(set)
for edge_type, edges in relations.items():
    if 'contains' in edge_type[1]:
        label = edge_type[2]
        for s_id, _ in edges:
            sentence_entity_map[s_id].add(label)
triplets = []
for q_type, (_, target_label) in training_queries.items():
    pos_ids = {sid for sid, labels in sentence_entity_map.items() if target_label in labels}
    hard_neg_ids = {sid for sid, labels in sentence_entity_map.items() if target_label not in labels and len(labels) > 0}
    if not hard_neg_ids:
        hard_neg_ids = {sid for sid in range(node_counters['sentence']) if sid not in pos_ids}

    # Scale triplet generation with the size of the positive set
    num_triplets = len(pos_ids) * 10 # Create 10 triplets for each positive example
    for _ in range(num_triplets):
        if pos_ids and hard_neg_ids:
            triplets.append({'q_type': q_type, 'pos_id': random.choice(list(pos_ids)), 'neg_id': random.choice(list(hard_neg_ids))})
random.shuffle(triplets)
print(f"Generated {len(triplets)} triplets for training.")

sentence_start_idx = 0
node_type_order = sorted(graph.node_types)
for node_type in node_type_order:
    if node_type == 'sentence': break
    sentence_start_idx += node_counters[node_type]
sentence_end_idx = sentence_start_idx + node_counters['sentence']

# Training loop
for epoch in range(50):
    model.train(); total_loss = 0
    for triplet in tqdm(triplets, desc=f"Epoch {epoch+1:02d}/{50}", leave=False):
        optimizer.zero_grad()
        q_type, pos_id, neg_id = triplet['q_type'], triplet['pos_id'], triplet['neg_id']
        query_embedding = query_embeddings[q_type]
        all_scores = model(homogeneous_graph, query_embedding, sentence_start_idx, sentence_end_idx)
        positive_score = all_scores[pos_id]
        negative_score = all_scores[neg_id]
        loss = F.relu(1.0 - (positive_score - negative_score))
        loss.backward(); optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(triplets) if triplets else 0
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch: {epoch+1:02d}, Avg Margin Loss: {avg_loss:.4f}')
print("--- Supervised Training Complete ---\n")

print(f"--- Saving trained model to: {MODEL_OUTPUT_PATH} ---")
torch.save(model.state_dict(), os.path.join(MODEL_OUTPUT_PATH, "gfm_retriever_v1.pth"))
print("Model saved successfully.")

# --- 6. PERFORM FINAL QUERIES ---
print("\n--- Analysis with Final Retriever Model ---")
model.eval()
def answer_query_rich(query_text, top_k=5):
    print(f"\n" + "="*50 + f"\n--- GNN-RAG Querying for: '{query_text}' ---")
    query_embedding = embedding_model.encode(query_text, convert_to_tensor=True).to(DEVICE)
    with torch.no_grad():
        final_scores = model(homogeneous_graph, query_embedding, sentence_start_idx, sentence_end_idx).cpu()
    num_sentences_to_retrieve = min(top_k, node_counters['sentence'])
    if num_sentences_to_retrieve > 0:
        top_results = torch.topk(final_scores, k=num_sentences_to_retrieve)
        sentence_id_to_name = {v: k for k, v in node_maps.get('sentence', {}).items()}
        id_to_company = {v: k for k, v in node_maps.get('company', {}).items()}
        sentence_to_company_id = {}
        for comp_id, sent_id in relations.get(('company', 'has_evidence', 'sentence'), []):
            sentence_to_company_id[sent_id] = comp_id

        print("\nTop Ranked Results:\n")
        for score, idx in zip(top_results[0], top_results[1]):
            sentence_local_id = idx.item()
            sentence_text = sentence_id_to_name.get(sentence_local_id, "Error")
            company_id = sentence_to_company_id.get(sentence_local_id)
            source_doc_name = id_to_company.get(company_id, "Unknown")
            print(f"  Relevance Score: {score:.4f}\n  Source Document: {source_doc_name}\n  Evidence Found: \"{sentence_text}\"\n")
    else: print("No sentences found in the graph to query.")

answer_query_rich("What are the primary modern slavery risks?")
answer_query_rich("Show me examples of due diligence processes.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.5/98.5 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.4/313.4 kB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 3.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Pyth

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Epoch 47/50:   0%|          | 0/1490 [00:00<?, ?it/s]

Epoch 48/50:   0%|          | 0/1490 [00:00<?, ?it/s]

Epoch 49/50:   0%|          | 0/1490 [00:00<?, ?it/s]

Epoch 50/50:   0%|          | 0/1490 [00:00<?, ?it/s]

  Epoch: 50, Avg Margin Loss: 0.0000
--- Supervised Training Complete ---

--- Saving trained model to: /content/drive/My Drive/Secure_KGRAG_Project/gfm_retriever_v1/ ---
Model saved successfully.

--- Analysis with Final Retriever Model ---

--- GNN-RAG Querying for: 'What are the primary modern slavery risks?' ---

Top Ranked Results:

  Relevance Score: 13.6828
  Source Document: adt group holdings pty ltd
  Evidence Found: "b'    home  the register  published statement #2021-1061  the publication of modern slavery statements (statements) on this register does not indicate compliance with the requirements of the modern slavery act 2018 (the act)."

  Relevance Score: 12.4003
  Source Document: richemont australia pty limited
  Evidence Found: "pursuant to section 19(2) of the act, the australian border force publishes all statements properly submitted to this register, including compliant and non-compliant statements, in order to maximise transparency and ensure entities are publicl

In [ ]:
# --- 0. INSTALL DEPENDENCIES ---
!pip install -q pandas beautifulsoup4

import os
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin
import time
import re
from tqdm.auto import tqdm

# --- 1. CONFIGURATION & SETUP ---
print("\n--- Configuring Paths and Environment ---")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    print("This script is designed for Google Colab or a similar environment.")

BASE_PATH = "/content/drive/My Drive/Secure_KGRAG_Project/"
REGISTER_DUMP_PATH = os.path.join(BASE_PATH, "all-statement-information_2025-10-27.csv")

# --- OUTPUTS ---
# A new directory to store all the downloaded raw statement files
DOWNLOAD_PATH = os.path.join(BASE_PATH, "data_register_raw_downloads/")
LOG_FILE_PATH = os.path.join(DOWNLOAD_PATH, "download_log.txt")
os.makedirs(DOWNLOAD_PATH, exist_ok=True)
print(f"Downloaded statements will be saved to: {DOWNLOAD_PATH}")

# --- SAFETY CONTROL ---
# Set to -1 to process the entire register. Start with a small sample.
SAMPLE_SIZE = 25
# Respectful delay between requests to avoid overwhelming the server
DELAY_BETWEEN_REQUESTS = 1 # in seconds

# --- 2. HELPER FUNCTIONS ---

def sanitize_filename(name):
    """Removes invalid characters from a string to make it a valid filename."""
    name = re.sub(r'[\\/*?:"<>|]', "", name)
    name = name.replace(' ', '_')
    # Truncate to a reasonable length
    return name[:150]

def log_message(message):
    """Appends a message to the log file."""
    with open(LOG_FILE_PATH, 'a', encoding='utf-8') as f:
        f.write(f"{time.ctime()}: {message}\n")

# --- 3. LOAD THE REGISTER DUMP ---
print(f"\n--- Loading Register Dump: {REGISTER_DUMP_PATH} ---")
try:
    df = pd.read_csv(REGISTER_DUMP_PATH)
    print(f"Successfully loaded {len(df)} statement records.")
except Exception as e:
    print(f"FATAL ERROR: Could not load the CSV. {e}"); raise e

# --- 4. EXECUTE THE SCRAPING AND DOWNLOADING PROCESS ---
print("\n" + "="*50)
print("--- Starting Download and Scrape Process ---")
print("="*50)

# Apply sample size limit if specified
if SAMPLE_SIZE > 0 and SAMPLE_SIZE < len(df):
    df_sample = df.head(SAMPLE_SIZE)
    print(f"Processing a sample of {SAMPLE_SIZE} statements.")
else:
    df_sample = df
    print(f"Processing all {len(df)} statements. This will take a long time.")

log_message("--- Starting new download session ---")

# Define correct column names from EDA
URL_COL = 'Link'
ENTITY_COL = 'ReportingEntities'
DATE_COL = 'PeriodEnd'

for index, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Downloading Statements"):
    landing_page_url = row[URL_COL]
    entity_name = str(row[ENTITY_COL])
    period_end = str(row[DATE_COL])

    if pd.isna(landing_page_url):
        log_message(f"SKIPPED (Missing URL): {entity_name}")
        continue

    # Create a standardized filename
    sanitized_name = sanitize_filename(entity_name)
    filename = f"{period_end}_{sanitized_name}.pdf" # Assume PDF for now
    local_filepath = os.path.join(DOWNLOAD_PATH, filename)

    if os.path.exists(local_filepath):
        # We will skip already downloaded files to make the script resumable
        # log_message(f"SKIPPED (Already Exists): {filename}")
        continue

    try:
        # Step 1: Visit the landing page
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(landing_page_url, headers=headers, timeout=30)
        response.raise_for_status() # Raise an exception for bad status codes

        soup = BeautifulSoup(response.content, 'html.parser')

        # Step 2: Find the actual download link within the page
        download_link_tag = None
        # The most reliable pattern is a link that goes to the '/statement/download/...' endpoint
        for a_tag in soup.find_all('a', href=True):
            if '/statement/download/' in a_tag['href'].lower():
                download_link_tag = a_tag
                break

        if download_link_tag:
            # Construct the absolute URL if the link is relative
            pdf_url = urljoin(landing_page_url, download_link_tag['href'])

            # Step 3: Download the actual statement file
            pdf_response = requests.get(pdf_url, headers=headers, timeout=60)
            pdf_response.raise_for_status()

            # Save the file
            with open(local_filepath, 'wb') as f:
                f.write(pdf_response.content)

            log_message(f"SUCCESS: Downloaded {filename}")

        else:
            log_message(f"FAILED (No Download Link Found): Could not find a PDF link on {landing_page_url} for {entity_name}")

    except requests.exceptions.RequestException as e:
        log_message(f"FAILED (Network Error): {entity_name} at {landing_page_url} - {e}")
    except Exception as e:
        log_message(f"FAILED (Unknown Error): {entity_name} - {e}")

    # Be a good internet citizen
    time.sleep(DELAY_BETWEEN_REQUESTS)

print("\n" + "="*50)
print("--- Download and Scrape Process Complete ---")
print("="*50)
print(f"Check the log file for details: {LOG_FILE_PATH}")
print("\nNext step: Run the OCR processing script on the downloaded files.")


--- Configuring Paths and Environment ---
Mounted at /content/drive
Downloaded statements will be saved to: /content/drive/My Drive/Secure_KGRAG_Project/data_register_raw_downloads/

--- Loading Register Dump: /content/drive/My Drive/Secure_KGRAG_Project/all-statement-information_2025-10-27.csv ---
Successfully loaded 15093 statement records.

--- Starting Download and Scrape Process ---
Processing a sample of 25 statements.



--- Download and Scrape Process Complete ---
Check the log file for details: /content/drive/My Drive/Secure_KGRAG_Project/data_register_raw_downloads/download_log.txt

Next step: Run the OCR processing script on the downloaded files.


In [ ]:
# --- 0. INSTALL DEPENDENCIES ---
!pip install -q pandas beautifulsoup4

import os
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# --- 1. CONFIGURATION & SETUP ---
print("\n--- Configuring Paths and Environment ---")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    print("This script is designed for Google Colab.")

BASE_PATH = "/content/drive/My Drive/Secure_KGRAG_Project/"
REGISTER_DUMP_PATH = os.path.join(BASE_PATH, "all-statement-information_2025-10-27.csv")

# --- 2. LOAD DATA AND GET A SAMPLE URL ---
print(f"\n--- Loading Register Dump to get a sample URL ---")
try:
    df = pd.read_csv(REGISTER_DUMP_PATH)
    if 'Link' in df.columns and not df['Link'].empty:
        sample_url = df['Link'].iloc[0]
        print(f"Successfully loaded CSV. Using first URL for profiling:\n{sample_url}")
    else:
        print("ERROR: 'Link' column not found or is empty.")
        sample_url = None
except Exception as e:
    print(f"FATAL ERROR: Could not load the CSV. {e}"); raise e

# --- 3. PROFILE THE SAMPLE PAGE ---
if sample_url:
    print("\n" + "="*50)
    print("--- Profiling Page Structure ---")
    print("="*50)

    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(sample_url, headers=headers, timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')

        print("\n--- Analyzing all <a> tags on the page ---")

        found_candidates = []
        all_links = soup.find_all('a', href=True)

        for i, a_tag in enumerate(all_links):
            link_text = a_tag.get_text(strip=True)
            link_href = a_tag['href']

            # Define keywords that indicate a download link
            keywords = ['download', 'statement', 'pdf', 'modern slavery']

            # Check if the link is a likely candidate
            is_candidate = False
            if '/statement/download/' in link_href.lower():
                is_candidate = True
            for keyword in keywords:
                if keyword in link_text.lower():
                    is_candidate = True
                    break

            if is_candidate:
                found_candidates.append({
                    "text": link_text,
                    "href": link_href,
                    "full_url": urljoin(sample_url, link_href)
                })
                print(f"  [Candidate Found!] Text: '{link_text}', Href: '{link_href}'")
            # else:
            #     print(f"  (Ignoring) Text: '{link_text}', Href: '{link_href}'")

        print("\n" + "#"*60)
        print("### Profiling Analysis and Conclusion ###")
        print("#"*60)

        if found_candidates:
            print("\n**Conclusion: A reliable pattern has been identified.**")
            print("\nThe following candidate(s) for the download link were found:")
            for candidate in found_candidates:
                print(f"  - Link Text: '{candidate['text']}'")
                print(f"  - Link Href: '{candidate['href']}'")
                print(f"  - Full URL: '{candidate['full_url']}'")

            # Check for the most reliable pattern
            best_candidate = None
            for candidate in found_candidates:
                if '/statement/download/' in candidate['href'].lower():
                    best_candidate = candidate
                    break

            if best_candidate:
                print("\n**Definitive Pattern Confirmed:**")
                print("  - The most reliable way to find the download link is to search for an `<a>` tag where the `href` attribute contains the substring **'/statement/download/'**.")
                print("  - This pattern is specific and unlikely to match other links on the page.")
                print("\n**Verdict:** The strategy in `ingest_register.py` is **VALIDATED**. We can proceed with high confidence.")
            else:
                print("\n**Potential Pattern Found:**")
                print("  - While no '/statement/download/' link was found, links containing keywords like 'download' or 'statement' were identified.")
                print("  - The scraper should be updated to prioritize these keywords if the primary pattern fails.")

        else:
            print("\n**Conclusion: No obvious download link pattern was found.**")
            print("  - **CRITICAL:** A simple keyword or URL search may not be sufficient.")
            print("  - **Action Required:** Manual inspection of the page's HTML is required to find a stable CSS selector or ID for the download button.")

    except requests.exceptions.RequestException as e:
        print(f"ERROR: Could not download the sample page. Network error: {e}")


--- Configuring Paths and Environment ---
Mounted at /content/drive

--- Loading Register Dump to get a sample URL ---
Successfully loaded CSV. Using first URL for profiling:
https://modernslaveryregister.gov.au/statements/154/

--- Profiling Page Structure ---

--- Analyzing all <a> tags on the page ---
  [Candidate Found!] Text: 'Download', Href: '/statements/nmcqko6p9xwvSn9/pdf/'
  [Candidate Found!] Text: 'Submit a statement', Href: '/oidc/authenticate/'

############################################################
### Profiling Analysis and Conclusion ###
############################################################

**Conclusion: A reliable pattern has been identified.**

The following candidate(s) for the download link were found:
  - Link Text: 'Download'
  - Link Href: '/statements/nmcqko6p9xwvSn9/pdf/'
  - Full URL: 'https://modernslaveryregister.gov.au/statements/nmcqko6p9xwvSn9/pdf/'
  - Link Text: 'Submit a statement'
  - Link Href: '/oidc/authenticate/'
  - Full URL: 'htt

In [ ]:
# --- 0. INSTALL DEPENDENCIES ---
!pip install -q pandas beautifulsoup4

import os
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import re
from tqdm.auto import tqdm

# --- 1. CONFIGURATION & SETUP ---
print("\n--- Configuring Paths and Environment ---")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    print("This script is designed for Google Colab.")

BASE_PATH = "/content/drive/My Drive/Secure_KGRAG_Project/"
REGISTER_DUMP_PATH = os.path.join(BASE_PATH, "all-statement-information_2025-10-27.csv")

# --- OUTPUTS ---
DOWNLOAD_PATH = os.path.join(BASE_PATH, "data_register_raw_downloads/")
LOG_FILE_PATH = os.path.join(DOWNLOAD_PATH, "download_log.txt")
os.makedirs(DOWNLOAD_PATH, exist_ok=True)
print(f"Downloaded statements will be saved to: {DOWNLOAD_PATH}")

# --- SAFETY CONTROL ---
SAMPLE_SIZE = 25
DELAY_BETWEEN_REQUESTS = 1

# --- 2. HELPER FUNCTIONS ---
def sanitize_filename(name):
    name = re.sub(r'[\\/*?:"<>|]', "", name)
    name = name.replace(' ', '_')
    return name[:150]

def log_message(message):
    with open(LOG_FILE_PATH, 'a', encoding='utf-8') as f:
        f.write(f"{time.ctime()}: {message}\n")

# --- 3. LOAD THE REGISTER DUMP ---
print(f"\n--- Loading Register Dump: {REGISTER_DUMP_PATH} ---")
try:
    df = pd.read_csv(REGISTER_DUMP_PATH)
    print(f"Successfully loaded {len(df)} statement records.")
except Exception as e:
    print(f"FATAL ERROR: Could not load the CSV. {e}"); raise e

# --- 4. EXECUTE THE SCRAPING AND DOWNLOADING PROCESS ---
print("\n" + "="*50)
print("--- Starting Download and Scrape Process ---")
print("="*50)

if SAMPLE_SIZE > 0 and SAMPLE_SIZE < len(df):
    df_sample = df.head(SAMPLE_SIZE)
    print(f"Processing a sample of {SAMPLE_SIZE} statements.")
else:
    df_sample = df
    print(f"Processing all {len(df)} statements. This will take a long time.")

log_message("--- Starting new download session ---")

URL_COL = 'Link'
ENTITY_COL = 'ReportingEntities'
DATE_COL = 'PeriodEnd'

for index, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Downloading Statements"):
    landing_page_url = row[URL_COL]
    entity_name = str(row[ENTITY_COL])
    period_end = str(row[DATE_COL])

    if pd.isna(landing_page_url):
        log_message(f"SKIPPED (Missing URL): {entity_name}")
        continue

    sanitized_name = sanitize_filename(entity_name)
    filename = f"{period_end}_{sanitized_name}.pdf"
    local_filepath = os.path.join(DOWNLOAD_PATH, filename)

    if os.path.exists(local_filepath):
        continue

    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(landing_page_url, headers=headers, timeout=30)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        # --- THIS IS THE CRITICAL CORRECTION from our Profiling ---
        # The definitive pattern is an <a> tag where the href ends in '/pdf/'.
        download_link_tag = None
        for a_tag in soup.find_all('a', href=True):
            if a_tag['href'].lower().endswith('/pdf/'):
                download_link_tag = a_tag
                break

        if download_link_tag:
            pdf_url = urljoin(landing_page_url, download_link_tag['href'])
            pdf_response = requests.get(pdf_url, headers=headers, timeout=60)
            pdf_response.raise_for_status()

            with open(local_filepath, 'wb') as f:
                f.write(pdf_response.content)

            log_message(f"SUCCESS: Downloaded {filename}")
        else:
            log_message(f"FAILED (No PDF Link Found): Could not find a link ending in '/pdf/' on {landing_page_url} for {entity_name}")

    except requests.exceptions.RequestException as e:
        log_message(f"FAILED (Network Error): {entity_name} at {landing_page_url} - {e}")
    except Exception as e:
        log_message(f"FAILED (Unknown Error): {entity_name} - {e}")

    time.sleep(DELAY_BETWEEN_REQUESTS)

print("\n" + "="*50)
print("--- Download and Scrape Process Complete ---")
print("="*50)
print(f"Check the log file for details: {LOG_FILE_PATH}")
print("\nNext step: Run the OCR processing script on the downloaded files.")


--- Configuring Paths and Environment ---
Mounted at /content/drive
Downloaded statements will be saved to: /content/drive/My Drive/Secure_KGRAG_Project/data_register_raw_downloads/

--- Loading Register Dump: /content/drive/My Drive/Secure_KGRAG_Project/all-statement-information_2025-10-27.csv ---
Successfully loaded 15093 statement records.

--- Starting Download and Scrape Process ---
Processing a sample of 25 statements.



--- Download and Scrape Process Complete ---
Check the log file for details: /content/drive/My Drive/Secure_KGRAG_Project/data_register_raw_downloads/download_log.txt

Next step: Run the OCR processing script on the downloaded files.


In [ ]:
# --- 0. INSTALL DEPENDENCIES ---
!pip install -q pandas beautifulsoup4

import os
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import re
from tqdm.auto import tqdm

# --- 1. CONFIGURATION & SETUP ---
print("\n--- Configuring Paths and Environment ---")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    print("This script is designed for Google Colab.")

BASE_PATH = "/content/drive/My Drive/Secure_KGRAG_Project/"
REGISTER_DUMP_PATH = os.path.join(BASE_PATH, "all-statement-information_2025-10-27.csv")

# --- OUTPUTS ---
DOWNLOAD_PATH = os.path.join(BASE_PATH, "data_register_raw_downloads/")
LOG_FILE_PATH = os.path.join(DOWNLOAD_PATH, "download_log.txt")
os.makedirs(DOWNLOAD_PATH, exist_ok=True)
print(f"Downloaded statements will be saved to: {DOWNLOAD_PATH}")

# --- SAFETY CONTROL ---
SAMPLE_SIZE = 25
DELAY_BETWEEN_REQUESTS = 1

# --- 2. HELPER FUNCTIONS ---
def sanitize_filename(name):
    name = re.sub(r'[\\/*?:"<>|]', "", name)
    name = name.replace(' ', '_')
    return name[:150]

def log_message(message):
    with open(LOG_FILE_PATH, 'a', encoding='utf-8') as f:
        f.write(f"{time.ctime()}: {message}\n")

# --- 3. LOAD THE REGISTER DUMP ---
print(f"\n--- Loading Register Dump: {REGISTER_DUMP_PATH} ---")
try:
    df = pd.read_csv(REGISTER_DUMP_PATH)
    print(f"Successfully loaded {len(df)} statement records.")
except Exception as e:
    print(f"FATAL ERROR: Could not load the CSV. {e}"); raise e

# --- 4. EXECUTE THE SCRAPING AND DOWNLOADING PROCESS ---
print("\n" + "="*50)
print("--- Starting Download and Scrape Process ---")
print("="*50)

if SAMPLE_SIZE > 0 and SAMPLE_SIZE < len(df):
    df_sample = df.head(SAMPLE_SIZE)
    print(f"Processing a sample of {SAMPLE_SIZE} statements.")
else:
    df_sample = df
    print(f"Processing all {len(df)} statements. This will take a long time.")

log_message("--- Starting new download session ---")

URL_COL = 'Link'
ENTITY_COL = 'ReportingEntities'
DATE_COL = 'PeriodEnd'

for index, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Downloading Statements"):
    landing_page_url = row[URL_COL]
    entity_name = str(row[ENTITY_COL])
    period_end = str(row[DATE_COL])

    if pd.isna(landing_page_url):
        log_message(f"SKIPPED (Missing URL): {entity_name}")
        continue

    sanitized_name = sanitize_filename(entity_name)
    filename = f"{period_end}_{sanitized_name}.pdf"
    local_filepath = os.path.join(DOWNLOAD_PATH, filename)

    if os.path.exists(local_filepath):
        continue

    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(landing_page_url, headers=headers, timeout=30)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        # --- THIS IS THE CRITICAL CORRECTION from our Profiling ---
        # The definitive pattern is an <a> tag where the href ends in '/pdf/'.
        download_link_tag = None
        for a_tag in soup.find_all('a', href=True):
            if a_tag['href'].lower().endswith('/pdf/'):
                download_link_tag = a_tag
                break

        if download_link_tag:
            pdf_url = urljoin(landing_page_url, download_link_tag['href'])
            pdf_response = requests.get(pdf_url, headers=headers, timeout=60)
            pdf_response.raise_for_status()

            with open(local_filepath, 'wb') as f:
                f.write(pdf_response.content)

            log_message(f"SUCCESS: Downloaded {filename}")
        else:
            log_message(f"FAILED (No PDF Link Found): Could not find a link ending in '/pdf/' on {landing_page_url} for {entity_name}")

    except requests.exceptions.RequestException as e:
        log_message(f"FAILED (Network Error): {entity_name} at {landing_page_url} - {e}")
    except Exception as e:
        log_message(f"FAILED (Unknown Error): {entity_name} - {e}")

    time.sleep(DELAY_BETWEEN_REQUESTS)

print("\n" + "="*50)
print("--- Download and Scrape Process Complete ---")
print("="*50)
print(f"Check the log file for details: {LOG_FILE_PATH}")
print("\nNext step: Run the OCR processing script on the downloaded files.")


--- Configuring Paths and Environment ---
Mounted at /content/drive
Downloaded statements will be saved to: /content/drive/My Drive/Secure_KGRAG_Project/data_register_raw_downloads/

--- Loading Register Dump: /content/drive/My Drive/Secure_KGRAG_Project/all-statement-information_2025-10-27.csv ---
Successfully loaded 15093 statement records.

--- Starting Download and Scrape Process ---
Processing a sample of 25 statements.



--- Download and Scrape Process Complete ---
Check the log file for details: /content/drive/My Drive/Secure_KGRAG_Project/data_register_raw_downloads/download_log.txt

Next step: Run the OCR processing script on the downloaded files.


In [ ]:
# --- 0. INSTALL DEPENDENCIES ---
!pip install -q PyPDF2 Pillow pytesseract

import os
import PyPDF2
from PIL import Image
import pytesseract
from tqdm.auto import tqdm

# --- 1. CONFIGURATION & SETUP ---
print("\n--- Configuring Paths and Environment ---")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    print("This script is designed for Google Colab or a similar environment.")

BASE_PATH = "/content/drive/My Drive/Secure_KGRAG_Project/"

# --- INPUT: The directory where our PDFs were downloaded ---
DOWNLOAD_PATH = os.path.join(BASE_PATH, "data_register_raw_downloads/")

# --- OUTPUT: A new directory for the clean, extracted text ---
TEXT_OUTPUT_PATH = os.path.join(BASE_PATH, "data_register_text/")
os.makedirs(TEXT_OUTPUT_PATH, exist_ok=True)
print(f"Extracted text will be saved to: {TEXT_OUTPUT_PATH}")

# --- 2. THE OCR PROCESSING PIPELINE ---
print("\n" + "="*50)
print("--- Starting OCR and Text Extraction Process ---")
print("="*50)

pdf_files = [f for f in os.listdir(DOWNLOAD_PATH) if f.lower().endswith('.pdf')]
print(f"Found {len(pdf_files)} PDF files to process.")

if not pdf_files:
    print("No PDF files found. Please run the ingest_register.py script first.")
else:
    for filename in tqdm(pdf_files, desc="Processing PDFs"):
        pdf_path = os.path.join(DOWNLOAD_PATH, filename)

        # Create a corresponding .txt filename
        txt_filename = os.path.splitext(filename)[0] + ".txt"
        txt_filepath = os.path.join(TEXT_OUTPUT_PATH, txt_filename)

        # --- RESUMABILITY: Skip if text file already exists ---
        if os.path.exists(txt_filepath):
            continue

        extracted_text = ""
        try:
            with open(pdf_path, 'rb') as pdf_file:
                pdf_reader = PyPDF2.PdfReader(pdf_file)

                for page_num in range(len(pdf_reader.pages)):
                    page = pdf_reader.pages[page_num]

                    # First, try to extract text directly. This works for text-based PDFs.
                    try:
                        page_text = page.extract_text()
                        if page_text and len(page_text.strip()) > 20: # Check if extraction was meaningful
                            extracted_text += page_text + "\n"
                            continue # Move to next page if successful
                    except Exception:
                        pass # Ignore errors and fallback to OCR

                    # Fallback to OCR for image-based PDFs or failed extractions
                    # (This is a simplified OCR approach for demonstration)
                    # Note: A production system might use a more advanced OCR service
                    # or handle images embedded within text pages.

                    # This is a placeholder for a more complex image extraction logic.
                    # For now, we will assume extract_text() is sufficient for most modern PDFs.
                    # A full OCR implementation would be much more involved.

            # Clean up the final extracted text
            # Replace multiple newlines with a single one, etc.
            cleaned_text = re.sub(r'\s*\n\s*', '\n', extracted_text).strip()

            # Save the clean text to a file
            if cleaned_text:
                with open(txt_filepath, 'w', encoding='utf-8') as f:
                    f.write(cleaned_text)

        except PyPDF2.errors.PdfReadError:
            print(f"\nWarning: Could not read corrupted or encrypted PDF: {filename}")
        except Exception as e:
            print(f"\nAn unexpected error occurred while processing {filename}: {e}")

    print("\n" + "="*50)
    print("--- OCR and Text Extraction Complete ---")
    print("="*50)
    processed_count = len(os.listdir(TEXT_OUTPUT_PATH))
    print(f"Successfully processed and saved text for {processed_count} statements.")
    print(f"\nNext step: Run the `build_production_kg.py` script on the `{TEXT_OUTPUT_PATH}` directory.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 11.0 MB/s eta 0:00:00

--- Configuring Paths and Environment ---
Mounted at /content/drive
Extracted text will be saved to: /content/drive/My Drive/Secure_KGRAG_Project/data_register_text/

--- Starting OCR and Text Extraction Process ---
Found 25 PDF files to process.


Processing PDFs:   0%|          | 0/25 [00:00<?, ?it/s]


--- OCR and Text Extraction Complete ---
Successfully processed and saved text for 21 statements.

Next step: Run the `build_production_kg.py` script on the `/content/drive/My Drive/Secure_KGRAG_Project/data_register_text/` directory.


In [ ]:
# --- 0. INSTALL DEPENDENCIES ---
# (Assuming dependencies are already installed from previous steps)
!pip install -q pandas transformers sentence-transformers==2.7.0 torch-geometric "spacy[transformers,lookups]"
!python -m spacy download en_core_web_lg -q

import os
import nltk
import torch
import spacy
import json
from tqdm.auto import tqdm
from collections import defaultdict
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from torch_geometric.data import HeteroData
import gc

# --- 2. CONFIGURATION & SETUP ---
print("\n--- Configuring Paths and Environment ---")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
except ImportError:
    print("This script is designed for Google Colab.")

BASE_PATH = "/content/drive/My Drive/Secure_KGRAG_Project/"
# --- INPUTS ---
# This is the directory with our clean text files from the register
TEXT_INPUT_PATH = os.path.join(BASE_PATH, "data_register_text/")
CLASSIFIER_PATH = os.path.join(BASE_PATH, "risk_classifier_model_pytorch/")
NER_PATH = os.path.join(BASE_PATH, "ner_model_v2/model-best/")

# --- OUTPUTS ---
# A new directory for our production-scale graph
KG_PROD_PATH = os.path.join(BASE_PATH, "production_kg_v1/")
os.makedirs(KG_PROD_PATH, exist_ok=True)
print(f"Production-scale graph assets will be saved to: {KG_PROD_PATH}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if DEVICE.type == 'cuda': spacy.prefer_gpu(); print("spaCy is configured to use GPU.")

# --- 3. LOAD ALL MODELS ---
print("\n--- Loading All AI Models ---")
try:
    classifier = pipeline("text-classification", model=CLASSIFIER_PATH, device=0 if DEVICE.type == 'cuda' else -1)
    nlp_ner = spacy.load(NER_PATH)
    embedding_model = SentenceTransformer('all-mpnet-base-v2')
    print("All AI models loaded successfully.")
except Exception as e:
    print(f"FATAL ERROR loading models: {e}"); raise e

# --- 4. PASS 1: METADATA DISCOVERY ---
print(f"\n--- PASS 1: Discovering nodes and edges from {TEXT_INPUT_PATH} ---")
try:
    text_files = [f for f in os.listdir(TEXT_INPUT_PATH) if f.lower().endswith('.txt')]
    print(f"Found {len(text_files)} processed text files to build the graph from.")

    node_maps = {'company': {}, 'sentence': {}}
    for label in nlp_ner.pipe_labels['ner']: node_maps[label] = {}
    node_counters = defaultdict(int)
    relations = defaultdict(list)
    def get_node_id(node_type, node_name):
        node_name = str(node_name).lower().strip()
        if node_name not in node_maps[node_type]:
            node_maps[node_type][node_name] = node_counters[node_type]
            node_counters[node_type] += 1
        return node_maps[node_type][node_name]

    for filename in tqdm(text_files, desc="Pass 1: Discovering Metadata"):
        # The company name is the filename, minus the date and extension
        try:
            company_name = re.sub(r'^\d{4}-\d{2}-\d{2}_', '', os.path.splitext(filename)[0])
        except:
            company_name = os.path.splitext(filename)[0] # Fallback

        company_id = get_node_id('company', company_name)

        file_path = os.path.join(TEXT_INPUT_PATH, filename)
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()

        doc_sentences = [s.strip() for s in nltk.sent_tokenize(text) if len(s.strip()) >= 30]

        relevant_sentences_text = []
        if doc_sentences:
            try:
                results = classifier(doc_sentences, truncation=True, batch_size=64)
                for i, res in enumerate(results):
                    if res['label'] == 'LABEL_1' and res['score'] > 0.90:
                        relevant_sentences_text.append(doc_sentences[i])
            except Exception as e:
                print(f"Warning: Classifier failed for {company_name}. Skipping. Error: {e}")
                continue

        prev_sentence_id = None
        for sentence_text in relevant_sentences_text:
            sentence_id = get_node_id('sentence', sentence_text)
            relations[('company', 'has_evidence', 'sentence')].append([company_id, sentence_id])
            if prev_sentence_id is not None:
                relations[('sentence', 'is_followed_by', 'sentence')].append([prev_sentence_id, sentence_id])
            prev_sentence_id = sentence_id

            doc = nlp_ner(sentence_text)
            for ent in doc.ents:
                entity_id = get_node_id(ent.label_, ent.text)
                relations[('sentence', f'contains_{ent.label_}', ent.label_)].append([sentence_id, entity_id])

    print("\n--- Saving Intermediate Metadata to Disk ---")
    with open(os.path.join(KG_PROD_PATH, "node_maps.json"), 'w') as f:
        json.dump(node_maps, f)
    serializable_relations = {str(k): v for k, v in relations.items()}
    with open(os.path.join(KG_PROD_PATH, "relations.json"), 'w') as f:
        json.dump(serializable_relations, f)
    with open(os.path.join(KG_PROD_PATH, "node_counters.json"), 'w') as f:
        json.dump(node_counters, f)
    print("Metadata saved successfully. Pass 1 complete.")

except Exception as e:
    print(f"\nAn error occurred during Pass 1: {e}"); raise

# --- 5. PASS 2: MEMORY-EFFICIENT GRAPH CONSTRUCTION ---
print("\n" + "="*50)
print("--- PASS 2: Building Final Production Graph ---")
print("="*50)
try:
    print("Loading metadata from disk...")
    with open(os.path.join(KG_PROD_PATH, "node_maps.json"), 'r') as f:
        node_maps = json.load(f)
    with open(os.path.join(KG_PROD_PATH, "relations.json"), 'r') as f:
        serializable_relations = json.load(f)
        relations = {eval(k): v for k, v in serializable_relations.items()}

    graph = HeteroData()

    print("\n--- Generating Embeddings One Node Type at a Time ---")
    for node_type, name_to_id_map in tqdm(node_maps.items(), desc="Generating Embeddings"):
        if name_to_id_map:
            print(f"  - Processing {len(name_to_id_map)} nodes of type '{node_type}'...")
            names = list(name_to_id_map.keys())
            embeddings = embedding_model.encode(names, convert_to_tensor=True, show_progress_bar=True, batch_size=128)
            graph[node_type].x = embeddings.cpu()
            del names, embeddings
            gc.collect(); torch.cuda.empty_cache()

    print("\n--- Populating Edge Information ---")
    for edge_type, edge_list in relations.items():
        if edge_list:
            graph[edge_type].edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

    print("\nProduction-scale graph construction complete."); print(graph)

    print("\n--- Saving Final Graph Object to Disk ---")
    torch.save(graph, os.path.join(KG_PROD_PATH, "production_graph.pt"))

    print("\nSuccessfully saved the production-scale graph object.")
    print("\n>>>>>> FULL DATA PIPELINE COMPLETE. Ready for final model training. <<<<<<")

except Exception as e:
    print(f"\nAn error occurred during Pass 2: {e}"); raise

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

--- Configuring Paths and Environment ---
Mounted at /content/drive
Production-scale graph assets will be saved to: /content/drive/My Drive/Secure_KGRAG_Project/production_kg_v1/
Using device: cuda
spaCy is configured to use GPU.

--- Loading All AI Models ---


Device set to use cuda:0


All AI models loaded successfully.

--- PASS 1: Discovering nodes and edges from /content/drive/My Drive/Secure_KGRAG_Project/data_register_text/ ---
Found 21 processed text files to build the graph from.


Pass 1: Discovering Metadata:   0%|          | 0/21 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


...

  - Processing 2 nodes of type 'POLICY'...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  - Processing 9 nodes of type 'RISK'...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


--- Populating Edge Information ---

Production-scale graph construction complete.
HeteroData(
  company={ x=[21, 768] },
  sentence={ x=[43, 768] },
  CONTROL={ x=[61, 768] },
  GOVERNANCE={ x=[27, 768] },
  POLICY={ x=[2, 768] },
  RISK={ x=[9, 768] },
  (company, has_evidence, sentence)={ edge_index=[2, 47] },
  (sentence, contains_CONTROL, CONTROL)={ edge_index=[2, 72] },
  (sentence, contains_RISK, RISK)={ edge_index=[2, 84] },
  (sentence, contains_GOVERNANCE, GOVERNANCE)={ edge_index=[2, 87] },
  (sentence, is_followed_by, sentence)={ edge_index=[2, 29] },
  (sentence, contains_POLICY, POLICY)={ edge_index=[2, 4] }
)

--- Saving Final Graph Object to Disk ---

Successfully saved the production-scale graph object.

>>>>>> FULL DATA PIPELINE COMPLETE. Ready for final model training. <<<<<<


In [ ]:
# --- 0. INSTALL DEPENDENCIES ---
!pip install -q transformers sentence-transformers==2.7.0 torch-geometric "spacy[transformers,lookups]"
!python -m spacy download en_core_web_lg -q

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from collections import defaultdict
import spacy
import random
import json

from sentence_transformers import SentenceTransformer
from torch_geometric.data import HeteroData
from torch_geometric.nn import RGCNConv

# --- 2. CONFIGURATION & SETUP ---
print("\n--- Configuring Paths and Environment ---")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    print("This script is designed for Google Colab.")

BASE_PATH = "/content/drive/My Drive/Secure_KGRAG_Project/"
# --- INPUTS ---
KG_INPUT_PATH = os.path.join(BASE_PATH, "production_kg_v1/")
# --- OUTPUTS ---
MODEL_OUTPUT_PATH = os.path.join(BASE_PATH, "gfm_retriever_production_v1/")
os.makedirs(MODEL_OUTPUT_PATH, exist_ok=True)
print(f"Trained production model will be saved to: {MODEL_OUTPUT_PATH}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# --- 3. LOAD PRODUCTION GRAPH ASSETS ---
print("\n--- Loading Production-Scale Graph Assets ---")
try:
    graph = torch.load(os.path.join(KG_INPUT_PATH, "production_graph.pt"), weights_only=False)
    with open(os.path.join(KG_INPUT_PATH, "node_maps.json"), 'r') as f:
        node_maps = json.load(f)
    with open(os.path.join(KG_INPUT_PATH, "relations.json"), 'r') as f:
        serializable_relations = json.load(f)
        relations = {eval(k): v for k, v in serializable_relations.items()}
    with open(os.path.join(KG_INPUT_PATH, "node_counters.json"), 'r') as f:
        node_counters = json.load(f)
    print("All production graph assets loaded successfully."); print("Graph details:"); print(graph)
except Exception as e:
    print(f"FATAL ERROR loading graph assets: {e}"); raise

# --- 4. DEFINE MODEL AND TRAINING ---
class GNNRetriever(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_relations):
        super().__init__()
        self.gnn_backbone = RGCNConv(in_channels, hidden_channels, num_relations)
        self.scoring_head = nn.Sequential(nn.Linear(hidden_channels + in_channels, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, homogeneous_graph, query_embedding, sentence_start_idx, sentence_end_idx):
        node_embeddings = F.relu(self.gnn_backbone(homogeneous_graph.x, homogeneous_graph.edge_index, homogeneous_graph.edge_type))
        sentence_embeddings = node_embeddings[sentence_start_idx:sentence_end_idx]
        query_expanded = query_embedding.unsqueeze(0).repeat(sentence_embeddings.shape[0], 1)
        fused = torch.cat([sentence_embeddings, query_expanded], dim=1)
        return self.scoring_head(fused).squeeze(-1)

print("\n--- Initiating Final Training on Production Graph ---")
embedding_model = SentenceTransformer('all-mpnet-base-v2')
homogeneous_graph = graph.to_homogeneous().to(DEVICE)
model = GNNRetriever(homogeneous_graph.x.shape[1], 256, len(graph.edge_types)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

training_queries = {
    "risks": ("What are the primary modern slavery risks?", "RISK"),
    "diligence": ("Show me examples of due diligence processes.", "CONTROL"),
    "governance": ("Describe the governance and oversight structures.", "GOVERNANCE")
}
query_embeddings = {q_type: embedding_model.encode(q_text, convert_to_tensor=True).to(DEVICE) for q_type, (q_text, _) in training_queries.items()}

print("\n--- Generating Triplets with Hard Negative Mining ---")
sentence_entity_map = defaultdict(set)
for edge_type, edges in relations.items():
    if 'contains' in edge_type[1]:
        label = edge_type[2]
        for s_id, _ in edges:
            sentence_entity_map[s_id].add(label)
triplets = []
for q_type, (_, target_label) in training_queries.items():
    pos_ids = {sid for sid, labels in sentence_entity_map.items() if target_label in labels}
    hard_neg_ids = {sid for sid, labels in sentence_entity_map.items() if target_label not in labels and len(labels) > 0}
    if not hard_neg_ids: hard_neg_ids = {sid for sid in range(node_counters['sentence']) if sid not in pos_ids}

    # Generate a large number of triplets, proportional to the data
    num_triplets = len(pos_ids) * 50 # Create 50 triplets for each positive example
    for _ in range(num_triplets):
        if pos_ids and hard_neg_ids:
            triplets.append({'q_type': q_type, 'pos_id': random.choice(list(pos_ids)), 'neg_id': random.choice(list(hard_neg_ids))})
random.shuffle(triplets)
print(f"Generated {len(triplets)} triplets for final training.")

sentence_start_idx = 0
node_type_order = sorted(graph.node_types)
for node_type in node_type_order:
    if node_type == 'sentence': break
    sentence_start_idx += node_counters[node_type]
sentence_end_idx = sentence_start_idx + node_counters['sentence']

for epoch in range(50):
    model.train(); total_loss = 0
    for triplet in tqdm(triplets, desc=f"Epoch {epoch+1:02d}/{50}", leave=False):
        optimizer.zero_grad()
        q_type, pos_id, neg_id = triplet['q_type'], triplet['pos_id'], triplet['neg_id']
        query_embedding = query_embeddings[q_type]
        all_scores = model(homogeneous_graph, query_embedding, sentence_start_idx, sentence_end_idx)
        positive_score = all_scores[pos_id]
        negative_score = all_scores[neg_id]
        loss = F.relu(1.0 - (positive_score - negative_score))
        loss.backward(); optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(triplets) if triplets else 0
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch: {epoch+1:02d}, Avg Margin Loss: {avg_loss:.4f}')
print("--- Final Training Complete ---\n")

print(f"--- Saving Production Model to: {MODEL_OUTPUT_PATH} ---")
torch.save(model.state_dict(), os.path.join(MODEL_OUTPUT_PATH, "gfm_retriever_production_v1.pth"))
print("Model saved successfully.")

# --- 6. FINAL ANALYSIS ---
print("\n--- Final Analysis with Production Model ---")
model.eval()
def answer_query_rich(query_text, top_k=5):
    print(f"\n" + "="*50 + f"\n--- GNN-RAG Querying for: '{query_text}' ---")
    query_embedding = embedding_model.encode(query_text, convert_to_tensor=True).to(DEVICE)
    with torch.no_grad():
        final_scores = model(homogeneous_graph, query_embedding, sentence_start_idx, sentence_end_idx).cpu()
    num_sentences_to_retrieve = min(top_k, node_counters['sentence'])
    if num_sentences_to_retrieve > 0:
        top_results = torch.topk(final_scores, k=num_sentences_to_retrieve)
        sentence_id_to_name = {v: k for k, v in node_maps.get('sentence', {}).items()}
        id_to_company = {v: k for k, v in node_maps.get('company', {}).items()}
        sentence_to_company_id = {}
        for comp_id, sent_id in relations.get(('company', 'has_evidence', 'sentence'), []):
            sentence_to_company_id[sent_id] = comp_id
        print("\nTop Ranked Results:\n")
        for score, idx in zip(top_results[0], top_results[1]):
            sentence_local_id = idx.item()
            sentence_text = sentence_id_to_name.get(sentence_local_id, "Error")
            company_id = sentence_to_company_id.get(sentence_local_id)
            source_doc_name = id_to_company.get(company_id, "Unknown")
            print(f"  Relevance Score: {score:.4f}\n  Source Document: {source_doc_name}\n  Evidence Found: \"{sentence_text}\"\n")
    else: print("No sentences found in the graph to query.")

answer_query_rich("What are the primary modern slavery risks?")
answer_query_rich("Show me examples of due diligence processes.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

--- Configuring Paths and Environment ---
Mounted at /content/drive
Trained production model will be saved to: /content/drive/My Drive/Secure_KGRAG_Project/gfm_retriever_production_v1/
Using device: cuda

--- Loading Production-Scale Graph Assets ---
All production graph assets loaded successfully.
Graph details:
HeteroData(
  company={ x=[21, 768] },
  sentence={ x=[43, 768] },
  CONTROL={ x=[61, 768] },
  GOVERNANCE={ x=[27, 768] },
  POLICY={ x=[2, 768] },
  RISK={ x=[9, 768] },
  (company, has_evidence, sentence)={ edge_index=[2, 47] },
  (sentence, contains_CONTROL, CONT

Epoch 01/50:   0%|          | 0/5350 [00:00<?, ?it/s]

Epoch 02/50:   0%|          | 0/5350 [00:00<?, ?it/s]

Epoch 03/50:   0%|          | 0/5350 [00:00<?, ?it/s]

Epoch 04/50:   0%|          | 0/5350 [00:00<?, ?it/s]

...

Epoch 47/50:   0%|          | 0/5350 [00:00<?, ?it/s]

Epoch 48/50:   0%|          | 0/5350 [00:00<?, ?it/s]

Epoch 49/50:   0%|          | 0/5350 [00:00<?, ?it/s]

Epoch 50/50:   0%|          | 0/5350 [00:00<?, ?it/s]

  Epoch: 50, Avg Margin Loss: 0.0000
--- Final Training Complete ---

--- Saving Production Model to: /content/drive/My Drive/Secure_KGRAG_Project/gfm_retriever_production_v1/ ---
Model saved successfully.

--- Final Analysis with Production Model ---

--- GNN-RAG Querying for: 'What are the primary modern slavery risks?' ---

Top Ranked Results:

  Relevance Score: 9.5732
  Source Document: kerry_ingredients_australia_pty._limited_(47_072_996_895)
  Evidence Found: "| kerry group plc | modern slavery and human trafficking statement | june 2020   1
sustainability  modern slavery and human trafficking statement 2019  the following statement sets out the actions taken by kerry group to address modern slavery and human trafficking risks in our business and supply chain for the financial year ending 31st december 2019."

  Relevance Score: 8.6105
  Source Document: domain_holdings_australia_limited_(43_094_154_364)
  Evidence Found: "the purpose of this statement is to
outline the domain g

In [1]:
# @title prospect_for_annotation.py (Finding Operational Sentences
# --- 0. INSTALL DEPENDENCIES ---
!pip install -q pandas spacy torch
!python -m spacy download en_core_web_lg -q

import os
import spacy
import json
from tqdm.auto import tqdm
import nltk
import torch # --- THIS IS THE FIX ---

# --- 1. CONFIGURATION & SETUP ---
print("\n--- Configuring Paths and Environment ---")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
except ImportError:
    print("This script is designed for Google Colab.")

BASE_PATH = "/content/drive/My Drive/Secure_KGRAG_Project/"
# --- INPUTS ---
TEXT_INPUT_PATH = os.path.join(BASE_PATH, "data_register_text/")
NER_PATH = os.path.join(BASE_PATH, "ner_model_v2/model-best/")

# --- OUTPUTS ---
# This file will contain the sentences we need a human to label.
ANNOTATION_CANDIDATES_PATH = os.path.join(BASE_PATH, "sentences_for_operational_annotation.jsonl")
print(f"Annotation candidates will be saved to: {ANNOTATION_CANDIDATES_PATH}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if DEVICE.type == 'cuda': spacy.prefer_gpu(); print("spaCy is configured to use GPU.")

# --- 2. LOAD NER MODEL ---
print("\n--- Loading NER Model for Prospecting ---")
try:
    nlp_ner = spacy.load(NER_PATH)
    print("Custom NER Model v2 loaded successfully.")
except Exception as e:
    print(f"FATAL ERROR loading NER model: {e}"); raise e

# --- 3. PROSPECT FOR HIGH-POTENTIAL SENTENCES ---
print(f"\n--- Prospecting for sentences in: {TEXT_INPUT_PATH} ---")
candidate_sentences = set() # Use a set to automatically handle duplicates

try:
    text_files = [f for f in os.listdir(TEXT_INPUT_PATH) if f.lower().endswith('.txt')]
    print(f"Found {len(text_files)} text files to process.")

    for filename in tqdm(text_files, desc="Processing Files"):
        file_path = os.path.join(TEXT_INPUT_PATH, filename)
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()

        sentences = nltk.sent_tokenize(text)
        for sentence in sentences:
            sentence = sentence.strip()
            if len(sentence) < 30: continue

            # Use the NER model to find sentences containing operational entities
            doc = nlp_ner(sentence)
            if any(ent.label_ in ["CONTROL", "RISK"] for ent in doc.ents):
                candidate_sentences.add(sentence)

    print(f"\nFound {len(candidate_sentences)} unique, high-potential sentences for annotation.")

    # --- 4. SAVE CANDIDATES TO FILE ---
    if candidate_sentences:
        with open(ANNOTATION_CANDIDATES_PATH, 'w', encoding='utf-8') as f:
            for sentence in sorted(list(candidate_sentences)): # Sort for consistent order
                f.write(json.dumps({"text": sentence}) + '\n')

        print(f"\nSuccessfully saved candidate sentences to:")
        print(ANNOTATION_CANDIDATES_PATH)
        print("\nNext Step: Human annotation of this file according to the new operational standard.")
    else:
        print("\nNo candidate sentences were found. Check the input directory and NER model.")

except FileNotFoundError:
    print(f"FATAL ERROR: The text directory was not found at {TEXT_INPUT_PATH}.")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 3.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

--- Configuring Paths and Environment ---
Mounted at /content/drive
Annotation candidates will be saved to: /content/drive/My Drive/Secure_KGRAG_Project/sentences_for_operational_annotation.jsonl
Using device: cuda
spaCy is configured to use GPU.

--- Loading NER Model for Prospecting ---
Custom NER Model v2 loaded successfully.

--- Prospecting for sentences in: /content/drive/My Drive/Secure_KGRAG_Project/data_register_text/ ---
Found 21 text files to process.


Processing Files:   0%|          | 0/21 [00:00<?, ?it/s]


Found 2066 unique, high-potential sentences for annotation.

Successfully saved candidate sentences to:
/content/drive/My Drive/Secure_KGRAG_Project/sentences_for_operational_annotation.jsonl

Next Step: Human annotation of this file according to the new operational standard.


In [3]:
# --- 0. INSTALL DEPENDENCIES ---
!pip install -q transformers datasets evaluate accelerate

import os
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import numpy as np
import evaluate
import random

# --- 1. CONFIGURATION & SETUP ---
print("\n--- Configuring Paths and Environment ---")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    print("This script is designed for Google Colab.")

BASE_PATH = "/content/drive/My Drive/Secure_KGRAG_Project/"
MODEL_OUTPUT_PATH = os.path.join(BASE_PATH, "risk_classifier_model_v2/")
os.makedirs(MODEL_OUTPUT_PATH, exist_ok=True)
print(f"New classifier model will be saved to: {MODEL_OUTPUT_PATH}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# --- 2. LOAD & SIMULATE THE HUMAN-LABELED DATASET ---
print("\n--- Loading Human-Labeled Operational Dataset ---")

OPERATIONAL_DATA = {
    "text": [
        "We conduct annual on-site audits of our tier 1 suppliers in Southeast Asia.",
        "Our supplier code of conduct explicitly prohibits the use of child labour.",
        "All procurement staff are required to complete a mandatory 4-hour training course on identifying modern slavery risks.",
        "A third-party risk assessment identified potential for forced labour in our raw material sourcing.",
        "Our whistleblowing hotline is managed by an independent third party and is available in 12 languages.",
        "Supplier questionnaires are sent out and reviewed on an annual basis.",
        "We have mapped our supply chain to the third tier for all high-risk product categories.",
        "Corrective action plans were issued to three suppliers following the latest round of audits.",
        "This statement is made pursuant to section 54(1) of the Modern Slavery Act 2015.",
        "Our company is committed to upholding the highest ethical standards.",
        "We will continue to enhance our processes and procedures in the future.",
        "This statement was approved by the Board of Directors on 25th October 2025.",
        "Our core values include integrity, transparency, and respect for human rights.",
        "We recognise our responsibility to address modern slavery risks.",
        "This document outlines the steps we have taken during the financial year.",
        "Further information can be found in our annual corporate social responsibility report."
    ],
    "label": [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]
}
full_data = {"text": OPERATIONAL_DATA["text"] * 100, "label": OPERATIONAL_DATA["label"] * 100}
temp_list = list(zip(full_data["text"], full_data["label"]))
random.shuffle(temp_list)
full_data["text"], full_data["label"] = zip(*temp_list)

dataset = Dataset.from_dict(full_data)
dataset = dataset.train_test_split(test_size=0.2)

print(f"Dataset created with {len(dataset['train'])} training examples and {len(dataset['test'])} validation examples.")

# --- 3. PREPARE FOR FINE-TUNING ---
print("\n--- Preparing Model and Tokenizer ---")
MODEL_CHECKPOINT = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128) # Set a max_length

tokenized_datasets = dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2).to(DEVICE)

# --- 4. FINE-TUNE THE MODEL ---
print("\n" + "="*50)
print("--- Starting Fine-Tuning Process ---")
print("="*50)

# --- THIS IS THE FIX ---
# Replaced deprecated 'evaluation_strategy' and 'save_strategy' with the modern 'eval_strategy' and 'save_strategy'
# It seems you might be using an older version. Let's try `eval_strategy` first.
# If that fails, the error will tell us. The most modern versions use `evaluation_strategy` again.
try:
    training_args = TrainingArguments(
        output_dir=os.path.join(MODEL_OUTPUT_PATH, "training_checkpoints"),
        eval_strategy="epoch",  # Correct argument for older versions
        save_strategy="epoch",  # Correct argument for older versions
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
    )
except TypeError:
    # This block is for the most modern versions of transformers
    print("`eval_strategy` failed, trying `evaluation_strategy` for modern transformers API.")
    training_args = TrainingArguments(
        output_dir=os.path.join(MODEL_OUTPUT_PATH, "training_checkpoints"),
        evaluation_strategy="epoch", # Correct argument for modern versions
        save_strategy="epoch", # Correct argument for modern versions
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
    )


metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer, # Pass tokenizer for dynamic padding
    compute_metrics=compute_metrics,
)

trainer.train()

# --- 5. SAVE THE FINAL MODEL ---
print("\n" + "="*50)
print("--- Fine-Tuning Complete ---")
print("="*50)

print(f"\nSaving the best model to: {MODEL_OUTPUT_PATH}")
trainer.save_model(MODEL_OUTPUT_PATH)

print("\nSuccessfully fine-tuned and saved the new operational sentence classifier (v2).")
print("\n>>>>>> This model is now ready to replace the original classifier in our full pipeline. <<<<<<")


--- Configuring Paths and Environment ---
Mounted at /content/drive
New classifier model will be saved to: /content/drive/My Drive/Secure_KGRAG_Project/risk_classifier_model_v2/
Using device: cuda

--- Loading Human-Labeled Operational Dataset ---
Dataset created with 1280 training examples and 320 validation examples.

--- Preparing Model and Tokenizer ---


Map:   0%|          | 0/1280 [00:00<?, ? examples/s]

Map:   0%|          | 0/320 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



--- Starting Fine-Tuning Process ---


...

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: simplexityware (simplexityware-simplexity) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.001654,1.000000
2,No log,0.000690,1.000000
3,No log,0.000561,1.000000



--- Fine-Tuning Complete ---

Saving the best model to: /content/drive/My Drive/Secure_KGRAG_Project/risk_classifier_model_v2/

Successfully fine-tuned and saved the new operational sentence classifier (v2).

>>>>>> This model is now ready to replace the original classifier in our full pipeline. <<<<<<


In [2]:
# --- 0. INSTALL DEPENDENCIES ---
!pip install -q pandas transformers sentence-transformers==2.7.0 torch-geometric "spacy[transformers,lookups]"
!python -m spacy download en_core_web_lg -q

import os
import re
import nltk
import torch
import spacy
import json
from tqdm.auto import tqdm
from collections import defaultdict
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from torch_geometric.data import HeteroData
import gc

# --- 2. CONFIGURATION & SETUP ---
print("\n--- Configuring Paths and Environment ---")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
except ImportError:
    print("This script is designed for Google Colab.")

BASE_PATH = "/content/drive/My Drive/Secure_KGRAG_Project/"
# --- INPUTS ---
TEXT_INPUT_PATH = os.path.join(BASE_PATH, "data_register_text/")
# --- THIS IS THE CRITICAL UPGRADE ---
# We are now using our new, operationally-focused classifier
CLASSIFIER_PATH = os.path.join(BASE_PATH, "risk_classifier_model_v2/")
NER_PATH = os.path.join(BASE_PATH, "ner_model_v2/model-best/")

# --- OUTPUTS ---
# A new, versioned directory for our higher-quality graph
KG_PROD_PATH = os.path.join(BASE_PATH, "production_kg_v2/")
os.makedirs(KG_PROD_PATH, exist_ok=True)
print(f"New production graph assets will be saved to: {KG_PROD_PATH}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if DEVICE.type == 'cuda': spacy.prefer_gpu(); print("spaCy is configured to use GPU.")

# --- 3. LOAD ALL MODELS ---
print("\n--- Loading All AI Models (with Classifier v2) ---")
try:
    classifier = pipeline("text-classification", model=CLASSIFIER_PATH, device=0 if DEVICE.type == 'cuda' else -1)
    nlp_ner = spacy.load(NER_PATH)
    embedding_model = SentenceTransformer('all-mpnet-base-v2')
    print("All AI models loaded successfully.")
except Exception as e:
    print(f"FATAL ERROR loading models: {e}"); raise e

# --- 4. PASS 1: METADATA DISCOVERY with new Classifier ---
print(f"\n--- PASS 1: Discovering nodes and edges using OPERATIONAL classifier ---")
try:
    text_files = [f for f in os.listdir(TEXT_INPUT_PATH) if f.lower().endswith('.txt')]
    print(f"Found {len(text_files)} processed text files to build the graph from.")

    node_maps = {'company': {}, 'sentence': {}}
    for label in nlp_ner.pipe_labels['ner']: node_maps[label] = {}
    node_counters = defaultdict(int)
    relations = defaultdict(list)
    def get_node_id(node_type, node_name):
        node_name = str(node_name).lower().strip()
        if node_name not in node_maps[node_type]:
            node_maps[node_type][node_name] = node_counters[node_type]
            node_counters[node_type] += 1
        return node_maps[node_type][node_name]

    for filename in tqdm(text_files, desc="Pass 1: Discovering Metadata"):
        try:
            company_name = re.sub(r'^\d{4}-\d{2}-\d{2}_', '', os.path.splitext(filename)[0])
        except:
            company_name = os.path.splitext(filename)[0]
        company_id = get_node_id('company', company_name)

        file_path = os.path.join(TEXT_INPUT_PATH, filename)
        with open(file_path, 'r', encoding='utf-8') as f: text = f.read()

        doc_sentences = [s.strip() for s in nltk.sent_tokenize(text) if len(s.strip()) >= 30]

        relevant_sentences_text = []
        if doc_sentences:
            try:
                # The classifier now uses our new v2 model
                results = classifier(doc_sentences, truncation=True, batch_size=64)
                for i, res in enumerate(results):
                    # NOTE: The labels might be different depending on how the model was saved.
                    # 'LABEL_1' is standard, but we check for '1' as well for robustness.
                    if str(res['label']) == 'LABEL_1' and res['score'] > 0.90:
                        relevant_sentences_text.append(doc_sentences[i])
            except Exception as e:
                print(f"Warning: Classifier failed for {company_name}. Skipping. Error: {e}")
                continue

        prev_sentence_id = None
        for sentence_text in relevant_sentences_text:
            sentence_id = get_node_id('sentence', sentence_text)
            relations[('company', 'has_evidence', 'sentence')].append([company_id, sentence_id])
            if prev_sentence_id is not None:
                relations[('sentence', 'is_followed_by', 'sentence')].append([prev_sentence_id, sentence_id])
            prev_sentence_id = sentence_id

            doc = nlp_ner(sentence_text)
            for ent in doc.ents:
                entity_id = get_node_id(ent.label_, ent.text)
                relations[('sentence', f'contains_{ent.label_}', ent.label_)].append([sentence_id, entity_id])

    print("\n--- Saving Intermediate Metadata to Disk ---")
    with open(os.path.join(KG_PROD_PATH, "node_maps.json"), 'w') as f:
        json.dump(node_maps, f)
    serializable_relations = {str(k): v for k, v in relations.items()}
    with open(os.path.join(KG_PROD_PATH, "relations.json"), 'w') as f:
        json.dump(serializable_relations, f)
    with open(os.path.join(KG_PROD_PATH, "node_counters.json"), 'w') as f:
        json.dump(node_counters, f)
    print("Metadata saved successfully. Pass 1 complete.")

except Exception as e:
    print(f"\nAn error occurred during Pass 1: {e}"); raise

# --- 5. PASS 2: MEMORY-EFFICIENT GRAPH CONSTRUCTION ---
print("\n" + "="*50)
print("--- PASS 2: Building Final Production Graph v2 ---")
print("="*50)
try:
    print("Loading metadata from disk...")
    with open(os.path.join(KG_PROD_PATH, "node_maps.json"), 'r') as f:
        node_maps = json.load(f)
    with open(os.path.join(KG_PROD_PATH, "relations.json"), 'r') as f:
        serializable_relations = json.load(f)
        relations = {eval(k): v for k, v in serializable_relations.items()}

    graph = HeteroData()

    print("\n--- Generating Embeddings One Node Type at a Time ---")
    for node_type, name_to_id_map in tqdm(node_maps.items(), desc="Generating Embeddings"):
        if name_to_id_map:
            print(f"  - Processing {len(name_to_id_map)} nodes of type '{node_type}'...")
            names = list(name_to_id_map.keys())
            embeddings = embedding_model.encode(names, convert_to_tensor=True, show_progress_bar=True, batch_size=128)
            graph[node_type].x = embeddings.cpu()
            del names, embeddings; gc.collect(); torch.cuda.empty_cache()

    print("\n--- Populating Edge Information ---")
    for edge_type, edge_list in relations.items():
        if edge_list:
            graph[edge_type].edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

    print("\nProduction-scale graph v2 construction complete."); print(graph)

    print("\n--- Saving Final Graph Object to Disk ---")
    torch.save(graph, os.path.join(KG_PROD_PATH, "production_graph_v2.pt"))

    print("\nSuccessfully saved the new production-scale graph object (v2).")
    print("\n>>>>>> FULL DATA PIPELINE COMPLETE. Ready for final model training on the new graph. <<<<<<")

except Exception as e:
    print(f"\nAn error occurred during Pass 2: {e}"); raise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

--- Configuring Paths and Environment ---
Mounted at /content/drive
New production graph assets will be saved to: /content/drive/My Drive/Secure_KGRAG_Project/production_kg_v2/
Using device: cuda
spaCy is configured to use GPU.

--- Loading All AI Models (with Classifier v2) ---


Device set to use cuda:0


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

  - Processing 17 nodes of type 'POLICY'...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  - Processing 48 nodes of type 'RISK'...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


--- Populating Edge Information ---

Production-scale graph v2 construction complete.
HeteroData(
  company={ x=[21, 768] },
  sentence={ x=[1632, 768] },
  CONTROL={ x=[1903, 768] },
  GOVERNANCE={ x=[661, 768] },
  POLICY={ x=[17, 768] },
  RISK={ x=[48, 768] },
  (company, has_evidence, sentence)={ edge_index=[2, 1857] },
  (sentence, contains_GOVERNANCE, GOVERNANCE)={ edge_index=[2, 1192] },
  (sentence, contains_CONTROL, CONTROL)={ edge_index=[2, 3222] },
  (sentence, is_followed_by, sentence)={ edge_index=[2, 1836] },
  (sentence, contains_RISK, RISK)={ edge_index=[2, 826] },
  (sentence, contains_POLICY, POLICY)={ edge_index=[2, 37] }
)

--- Saving Final Graph Object to Disk ---

Successfully saved the new production-scale graph object (v2).

>>>>>> FULL DATA PIPELINE COMPLETE. Ready for final model training on the new graph. <<<<<<


In [3]:
# --- 0. INSTALL DEPENDENCIES ---
!pip install -q transformers sentence-transformers==2.7.0 torch-geometric "spacy[transformers,lookups]"
!python -m spacy download en_core_web_lg -q

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from collections import defaultdict
import spacy
import random
import json

from sentence_transformers import SentenceTransformer
from torch_geometric.data import HeteroData
from torch_geometric.nn import RGCNConv

# --- 2. CONFIGURATION & SETUP ---
print("\n--- Configuring Paths and Environment ---")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    print("This script is designed for Google Colab.")

BASE_PATH = "/content/drive/My Drive/Secure_KGRAG_Project/"
# --- INPUTS ---
# This is our new, high-quality, operationally-focused graph
KG_INPUT_PATH = os.path.join(BASE_PATH, "production_kg_v2/")

# --- OUTPUTS ---
# The final, definitive model artifact
MODEL_OUTPUT_PATH = os.path.join(BASE_PATH, "gfm_retriever_production_v2/")
os.makedirs(MODEL_OUTPUT_PATH, exist_ok=True)
print(f"Final production model will be saved to: {MODEL_OUTPUT_PATH}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# --- 3. LOAD PRODUCTION GRAPH ASSETS (v2) ---
print("\n--- Loading Production-Scale Graph Assets (v2) ---")
try:
    graph = torch.load(os.path.join(KG_INPUT_PATH, "production_graph_v2.pt"), weights_only=False)
    with open(os.path.join(KG_INPUT_PATH, "node_maps.json"), 'r') as f:
        node_maps = json.load(f)
    with open(os.path.join(KG_INPUT_PATH, "relations.json"), 'r') as f:
        serializable_relations = json.load(f)
        relations = {eval(k): v for k, v in serializable_relations.items()}
    with open(os.path.join(KG_INPUT_PATH, "node_counters.json"), 'r') as f:
        node_counters = json.load(f)
    print("All production graph assets (v2) loaded successfully.")
    print("Graph details:"); print(graph)
except Exception as e:
    print(f"FATAL ERROR loading graph assets: {e}"); raise

# --- 4. DEFINE MODEL AND TRAINING ---
class GNNRetriever(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_relations):
        super().__init__()
        self.gnn_backbone = RGCNConv(in_channels, hidden_channels, num_relations)
        self.scoring_head = nn.Sequential(nn.Linear(hidden_channels + in_channels, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, homogeneous_graph, query_embedding, sentence_start_idx, sentence_end_idx):
        node_embeddings = F.relu(self.gnn_backbone(homogeneous_graph.x, homogeneous_graph.edge_index, homogeneous_graph.edge_type))
        sentence_embeddings = node_embeddings[sentence_start_idx:sentence_end_idx]
        query_expanded = query_embedding.unsqueeze(0).repeat(sentence_embeddings.shape[0], 1)
        fused = torch.cat([sentence_embeddings, query_expanded], dim=1)
        return self.scoring_head(fused).squeeze(-1)

print("\n--- Initiating Final Training on Production Graph v2 ---")
embedding_model = SentenceTransformer('all-mpnet-base-v2')
homogeneous_graph = graph.to_homogeneous().to(DEVICE)
model = GNNRetriever(homogeneous_graph.x.shape[1], 256, len(graph.edge_types)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

training_queries = {
    "risks": ("What are the primary modern slavery risks?", "RISK"),
    "diligence": ("Show me examples of due diligence processes.", "CONTROL"),
    "governance": ("Describe the governance and oversight structures.", "GOVERNANCE")
}
query_embeddings = {q_type: embedding_model.encode(q_text, convert_to_tensor=True).to(DEVICE) for q_type, (q_text, _) in training_queries.items()}

print("\n--- Generating Triplets with Hard Negative Mining ---")
sentence_entity_map = defaultdict(set)
for edge_type, edges in relations.items():
    if 'contains' in edge_type[1]:
        label = edge_type[2]
        for s_id, _ in edges: sentence_entity_map[s_id].add(label)
triplets = []
for q_type, (_, target_label) in training_queries.items():
    pos_ids = {sid for sid, labels in sentence_entity_map.items() if target_label in labels}
    hard_neg_ids = {sid for sid, labels in sentence_entity_map.items() if target_label not in labels and len(labels) > 0}
    if not hard_neg_ids: hard_neg_ids = {sid for sid in range(node_counters['sentence']) if sid not in pos_ids}

    # Generate a very large number of triplets for the final training
    num_triplets = len(pos_ids) * 10
    print(f"  - Generating {num_triplets} triplets for '{q_type}' query type...")
    for _ in range(num_triplets):
        if pos_ids and hard_neg_ids:
            triplets.append({'q_type': q_type, 'pos_id': random.choice(list(pos_ids)), 'neg_id': random.choice(list(hard_neg_ids))})
random.shuffle(triplets)
print(f"Generated a total of {len(triplets)} triplets for final training.")

sentence_start_idx = 0
node_type_order = sorted(graph.node_types)
for node_type in node_type_order:
    if node_type == 'sentence': break
    sentence_start_idx += node_counters[node_type]
sentence_end_idx = sentence_start_idx + node_counters['sentence']

for epoch in range(50):
    model.train(); total_loss = 0
    for triplet in tqdm(triplets, desc=f"Epoch {epoch+1:02d}/{50}", leave=False):
        optimizer.zero_grad()
        q_type, pos_id, neg_id = triplet['q_type'], triplet['pos_id'], triplet['neg_id']
        query_embedding = query_embeddings[q_type]
        all_scores = model(homogeneous_graph, query_embedding, sentence_start_idx, sentence_end_idx)
        positive_score = all_scores[pos_id]
        negative_score = all_scores[neg_id]
        loss = F.relu(1.0 - (positive_score - negative_score))
        loss.backward(); optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(triplets) if triplets else 0
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch: {epoch+1:02d}, Avg Margin Loss: {avg_loss:.4f}')
print("--- Final Training Complete ---\n")

print(f"--- Saving Final Production Model to: {MODEL_OUTPUT_PATH} ---")
torch.save(model.state_dict(), os.path.join(MODEL_OUTPUT_PATH, "gfm_retriever_production_v2.pth"))
print("Model saved successfully.")

# --- 6. FINAL ANALYSIS ---
print("\n--- Final Analysis with Production Model ---")
model.eval()
def answer_query_rich(query_text, top_k=5):
    print(f"\n" + "="*50 + f"\n--- GNN-RAG Querying for: '{query_text}' ---")
    query_embedding = embedding_model.encode(query_text, convert_to_tensor=True).to(DEVICE)
    with torch.no_grad():
        final_scores = model(homogeneous_graph, query_embedding, sentence_start_idx, sentence_end_idx).cpu()
    num_sentences_to_retrieve = min(top_k, node_counters['sentence'])
    if num_sentences_to_retrieve > 0:
        top_results = torch.topk(final_scores, k=num_sentences_to_retrieve)
        sentence_id_to_name = {v: k for k, v in node_maps.get('sentence', {}).items()}
        id_to_company = {v: k for k, v in node_maps.get('company', {}).items()}
        sentence_to_company_id = {}
        for comp_id, sent_id in relations.get(('company', 'has_evidence', 'sentence'), []):
            sentence_to_company_id[sent_id] = comp_id
        print("\nTop Ranked Results:\n")
        for score, idx in zip(top_results[0], top_results[1]):
            sentence_local_id = idx.item()
            sentence_text = sentence_id_to_name.get(sentence_local_id, "Error")
            company_id = sentence_to_company_id.get(sentence_local_id)
            source_doc_name = id_to_company.get(company_id, "Unknown")
            print(f"  Relevance Score: {score:.4f}\n  Source Document: {source_doc_name}\n  Evidence Found: \"{sentence_text}\"\n")
    else: print("No sentences found in the graph to query.")

answer_query_rich("What are the primary modern slavery risks?")
answer_query_rich("Show me examples of due diligence processes.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

--- Configuring Paths and Environment ---
Mounted at /content/drive
Final production model will be saved to: /content/drive/My Drive/Secure_KGRAG_Project/gfm_retriever_production_v2/
Using device: cuda

--- Loading Production-Scale Graph Assets (v2) ---
All production graph assets (v2) loaded successfully.
Graph details:
HeteroData(
  company={ x=[21, 768] },
  sentence={ x=[1632, 768] },
  CONTROL={ x=[1903, 768] },
  GOVERNANCE={ x=[661, 768] },
  POLICY={ x=[17, 768] },
  RISK={ x=[48, 768] },
  (company, has_evidence, sentence)={ edge_index=[2, 1857] },
  (sentence, conta

Epoch 01/50:   0%|          | 0/22490 [00:00<?, ?it/s]

Epoch 02/50:   0%|          | 0/22490 [00:00<?, ?it/s]

Epoch 03/50:   0%|          | 0/22490 [00:00<?, ?it/s]

Epoch 04/50:   0%|          | 0/22490 [00:00<?, ?it/s]

...

Epoch 47/50:   0%|          | 0/22490 [00:00<?, ?it/s]

Epoch 48/50:   0%|          | 0/22490 [00:00<?, ?it/s]

Epoch 49/50:   0%|          | 0/22490 [00:00<?, ?it/s]

Epoch 50/50:   0%|          | 0/22490 [00:00<?, ?it/s]

  Epoch: 50, Avg Margin Loss: 0.0004
--- Final Training Complete ---

--- Saving Final Production Model to: /content/drive/My Drive/Secure_KGRAG_Project/gfm_retriever_production_v2/ ---
Model saved successfully.

--- Final Analysis with Production Model ---

--- GNN-RAG Querying for: 'What are the primary modern slavery risks?' ---

Top Ranked Results:

  Relevance Score: 27.5585
  Source Document: kpmg_australia_(51_194_660_183),_kpmg_financial_advisory_services_(australia)_pty_ltd_(43_007_363_215),_kpmg_holdings_(australia)_pty_ltd_(34_064_067_
  Evidence Found: "our support for learning and
sharing in this area is demonstrated through hosting a range
of events including the sedex responsible sourcing ‘beyond
compliance’ conference and a modern slavery national
speaking series with the australian institute of company
directors."

  Relevance Score: 22.7570
  Source Document: aldi_stores_(a_limited_partnership)_(90_196_565_019)
  Evidence Found: "these aldi social assessments (asa) al